# Data Versioning & Data Lineage

Let's see how to version data and keep track of the processing that produces them, with [DVC](https://dvc.org/). DVC separates data versioning, git tracking their metadata (`.dvc` files), from data storage.

The steps and commands are those of the lab and of the course's “End-to-end workflow” frame. So that the demo runs without an account, a bare git repository on this machine, `/tmp/remote.git`, stands for the lab's DagsHub repository, and a directory, `/tmp/dvc-store`, for its S3 storage.

## Setup

This cell installs DVC and its S3 extension, then moves into the project's directory, `~/dvc-project`.

In [1]:
!apt install tree
!pip install dvc dvc-s3
!mkdir -p ~/dvc-project
%cd ~/dvc-project







































































































































































































































































































































/root/dvc-project


## 1. Create the git repository

The project's git repository, linked to the “remote” one.

In [2]:
!git init --bare -b main /tmp/remote.git
!git init -b main
!git config user.name "Jeanne Durant"
!git config user.email "jeanne@durant.fr"
!git remote add origin /tmp/remote.git

## 2. Set up DVC and its remote storage

`dvc init` sets up the repository, and `dvc remote add -d` declares the default remote storage, the one `dvc push` and `dvc pull` use without `-r`. With DagsHub, `dvc remote modify` would add the server's address and, with `--local`, the credentials, in `.dvc/config.local`, which `.dvc/.gitignore` keeps out of git.

In [3]:
!dvc init
!dvc remote add -d origin /tmp/dvc-store

In [4]:
!cat .dvc/config .dvc/.gitignore

In [5]:
!git add .dvc
!git commit -m "Initialize DVC"
!git push origin main

## 3. Retrieve a first version of the data

[`dvc get`](https://dvc.org/doc/command-reference/get) downloads a file from a git or DVC repository, and `dvc add` has DVC track it: it copies it into the cache, writes `data.csv.dvc` and adds `data.csv` to `.gitignore`.

In [6]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [7]:
!dvc add data.csv

In [8]:
!cat data.csv.dvc
!tree .dvc/cache

`data.csv.dvc` only holds the file's md5, its size and its path: this small file is what git versions. The cache stores the data under their md5.

## 4. Save this version

We commit the files `dvc add` named, with the `v1` tag. `git push` only sends this metadata, and `dvc push` the data (81 MB), from the cache, to the storage.

In [9]:
!git add data.csv.dvc .gitignore
!git commit -m "Raw data"
!git tag v1

In [10]:
!git push origin main v1
!dvc push

## 5. Modify the data

We sort the lines of `data.csv` in reverse order, then save this new version like the first one, with the `v2` tag.

In [11]:
!sort -r data.csv -o data.csv

In [12]:
!dvc add data.csv

In [13]:
!cat data.csv.dvc
!tree .dvc/cache

In [14]:
!git add data.csv.dvc
!git commit -m "Data v2"
!git tag v2

In [15]:
!git push origin main v2
!dvc push

The md5 changed, not the size. The cache now holds both versions, each under its md5.

## 6. Back to the original data

The new version has a problem: we go back to the first one. `git restore --source v1` puts back the `data.csv.dvc` of `v1`, and `dvc checkout` the data it points to, from the cache.

In [16]:
!git restore --source v1 data.csv.dvc
!dvc checkout

In [17]:
!cat data.csv.dvc

In [18]:
!git add data.csv.dvc
!git commit -m "Back to data v1"
!git tag v3

In [19]:
!git push origin main v3
!dvc push

`dvc push` has nothing to send: the `v1` data are already on the storage, under the same md5.

## 7. A pipeline stage

The `upper.py` script copies a file in uppercase. `dvc stage add` makes it a pipeline stage, with its dependencies (the script, the data) and its output, and `dvc repro` runs it and writes `dvc.lock`.

In [20]:
%%writefile upper.py
from pathlib import Path
from sys import argv

text = Path(argv[1]).read_text(encoding="utf-8")
Path(argv[2]).write_text(text.upper(), encoding="utf-8")

Writing upper.py


In [21]:
!dvc stage add --name uppercase --deps upper.py --deps data.csv --outs data-upper.csv python upper.py data.csv data-upper.csv

In [22]:
!dvc repro

In [23]:
!git add upper.py dvc.yaml dvc.lock .gitignore
!git commit -m "Uppercase pipeline"
!git push origin main
!dvc push

In [24]:
!dvc repro
!cat dvc.lock

The second `dvc repro` skips the stage: nothing changed since the md5 in `dvc.lock`. `dvc.lock` says exactly from what and how `data-upper.csv` was produced: this is data lineage.

## 8. On another machine

A colleague joins the project: she clones the repository and retrieves the data with `dvc pull`. With DagsHub, she would first give the clone her credentials, which git didn't pass on.

In [25]:
!git clone /tmp/remote.git ~/other-machine

In [26]:
%cd ~/other-machine

/root/other-machine


In [27]:
!dvc pull

In [28]:
!ls
!dvc status
!dvc repro

The clone has `data.csv` (in its `v3` version) and `data-upper.csv`, without running the script again: the data and the pipeline are up to date.